# Lab 06 - Arquitetura Medallion e Delta Lake com Spark Streaming
Vamos implementar as camadas Bronze e Silver utilizando o formato Delta, que é o padrão industrial para Lakehouses.

**Objetivos:**
1. Gravar dados brutos de streaming em formato Delta (Bronze).
2. Transformar e gravar em uma tabela Delta refinada (Silver).
3. Inspecionar o histórico de transações ACID do Delta Lake.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# 1. Parar queries ativas anteriores
for stream in spark.streams.active:
    stream.stop()

# 2. Configurações de caminhos no Volume / DBFS
path_bronze = "/Volumes/workspace/default/checkpoint/lab06_delta/bronze"
path_silver = "/Volumes/workspace/default/checkpoint/lab06_delta/silver"
checkpoint_bronze = "/Volumes/workspace/default/checkpoint/lab06_checkpoints/bronze"
checkpoint_silver = "/Volumes/workspace/default/checkpoint/lab06_checkpoints/silver"

# 3. Limpeza de execuções anteriores
dbutils.fs.rm("/Volumes/workspace/default/checkpoint/lab06_delta", True)
dbutils.fs.rm("/Volumes/workspace/default/checkpoint/lab06_checkpoints", True)

In [0]:
# 4. Camada BRONZE: Ingestão Raw em Streaming para Delta Lake
inputPath = "/databricks-datasets/structured-streaming/events/"
jsonSchema = StructType([ 
    StructField("time", TimestampType(), True), 
    StructField("action", StringType(), True) 
])

df_raw = (spark.readStream
          .schema(jsonSchema)
          .option("maxFilesPerTrigger", 1)
          .json(inputPath))

query_bronze = (df_raw.writeStream
                .format("delta")
                .outputMode("append")
                .option("checkpointLocation", checkpoint_bronze)
                .trigger(availableNow=True)
                .start(path_bronze))

query_bronze.awaitTermination()
print("Camada Bronze processada com sucesso!")

In [0]:
# 5. Camada SILVER: Leitura do Stream Bronze -> Transformação -> Delta Silver
df_bronze = spark.readStream.format("delta").load(path_bronze)

df_silver = (df_bronze
             .withColumn("processamento_ts", current_timestamp())
             .filter(col("action").isNotNull()))

query_silver = (df_silver.writeStream
                .format("delta")
                .outputMode("append")
                .option("checkpointLocation", checkpoint_silver)
                .trigger(availableNow=True)
                .start(path_silver))

query_silver.awaitTermination()
print("Camada Silver processada com sucesso!")

In [0]:
# 6. Validando o Pipeline Medallion (Camada Silver)
df_silver_res = spark.read.format("delta").load(path_silver)
display(df_silver_res.orderBy(desc("processamento_ts")))

In [0]:
# 7. Auditoria do Delta Transaction Log (Histórico de transações ACID)
display(spark.sql(f"DESCRIBE HISTORY delta.`{path_silver}`"))

In [0]:
# 8. Encerramento gracioso das queries
query_bronze.stop()
query_silver.stop()
print("Queries das camadas Bronze e Silver encerradas com sucesso.")